In [9]:
import json
import re
from pathlib import Path

import pandas as pd

In [10]:
input_dir = Path(r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\csv_output\with_NA")
output_dir = Path(r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\With_na")

# Create output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)


In [11]:
labels_structure = {
    "MT": [],
    "LY": [],
    "SP": ["it"], #os
    "ID": [],
    "NA": ["ne", "sr", "nb"], #on
    "HI": ["re"], #oh
    "IN": ["en", "ra", "dtp", "fi", "lt"], #oi
    "OP": ["rv", "ob", "rs", "av"], #oo
    "IP": ["ds", "ed"], #oe
}

# Create set of main and sub-level labels
valid_labels = set(labels_structure.keys())

for sub_labels in labels_structure.values():
    valid_labels.update(sub_labels)


In [12]:
def consolidate_labels(group):
    all_labels = set()

    # Collect labels from Turku_NLP
    for val in group["Turku_NLP"]:
        if val and val != "":
            labels = [label.strip() for label in str(val).split(";")]
            all_labels.update(labels)

    # Collect labels from Turku_NLP_sub
    for val in group["Turku_NLP_sub"]:
        if val and val != "":
            labels = [label.strip() for label in str(val).split(";")]
            all_labels.update(labels)

    # Remove non-alphabetic characters
    all_labels = [
        re.sub(r"[^a-zA-Z\s]", "", label).strip()
        for label in all_labels
    ]

    # Remove empty labels
    all_labels = [label for label in all_labels if label]

    # Keep only valid labels
    all_labels = [
        label for label in all_labels
        if label in valid_labels
    ]

    # Sort alphabetically
    all_labels = sorted(all_labels)

    # Join with spaces
    return " ".join(all_labels)

In [13]:
csv_files = list(input_dir.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files.\n")


for csv_file in csv_files:

    print("=" * 70)
    print(f"Processing: {csv_file.name}")

    try:
        # ----------------------------------------------------
        # Load CSV
        # ----------------------------------------------------
        df = pd.read_csv(
            csv_file,
            keep_default_na=False,
            na_values=[""]
        )

        # ----------------------------------------------------
        # Check required columns
        # ----------------------------------------------------
        required_columns = {
            "id",
            "text",
            "Turku_NLP",
            "Turku_NLP_sub"
        }

        missing_columns = required_columns - set(df.columns)

        if missing_columns:
            print(
                f"  ⚠ Skipping file. Missing columns: "
                f"{sorted(missing_columns)}"
            )
            continue

        # ----------------------------------------------------
        # Consolidate by ID
        # ----------------------------------------------------
        consolidated = (
            df.groupby("id")
            .agg(
                {
                    "text": "first"
                }
            )
            .reset_index()
        )

        # ----------------------------------------------------
        # Add consolidated labels
        # ----------------------------------------------------
        labels_by_id = df.groupby("id").apply(
            consolidate_labels
        )

        consolidated["label"] = (
            consolidated["id"]
            .map(labels_by_id)
            .fillna("")
        )

        # ----------------------------------------------------
        # Create JSONL records
        # ----------------------------------------------------
        records = consolidated.to_dict("records")

        # ----------------------------------------------------
        # Output filename
        #
        # Example:
        # results_Final.CSV
        #     ↓
        # results_Final.jsonl
        # ----------------------------------------------------
        output_file = output_dir / f"{csv_file.stem}.jsonl"

        # ----------------------------------------------------
        # Save JSONL
        # ----------------------------------------------------
        with open(output_file, "w", encoding="utf-8") as f:
            for record in records:
                json.dump(
                    record,
                    f,
                    ensure_ascii=False
                )
                f.write("\n")

        # ----------------------------------------------------
        # Print information
        # ----------------------------------------------------
        print(f"  ✓ CSV rows:       {len(df)}")
        print(f"  ✓ Unique IDs:     {len(consolidated)}")
        print(f"  ✓ Output:         {output_file}")

        # Show first 3 records
        print("\n  Sample of first 3 records:")

        for i, record in enumerate(records[:3]):
            print(f"\n  Record {i + 1}:")
            print(f"    ID: {record['id']}")
            print(f"    Label: {record['label']}")
            print(
                f"    Text preview: "
                f"{str(record['text'])[:100]}..."
            )

    except Exception as e:
        print(f"  ✗ ERROR processing {csv_file.name}: {e}")


print("\n" + "=" * 70)
print("Finished processing all CSV files.")
print(f"JSONL files saved to: {output_dir}")

Found 4 CSV files.

Processing: Combined_hybrid.csv


C:\Users\alrazz\AppData\Local\Temp\ipykernel_13236\3797153780.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labels_by_id = df.groupby("id").apply(


  ✓ CSV rows:       9198
  ✓ Unique IDs:     6160
  ✓ Output:         C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\With_na\Combined_hybrid.jsonl

  Sample of first 3 records:

  Record 1:
    ID: 0003f44c9a50428a917087202a2fd3cc
    Label: 
    Text preview: مجموعه های ذخیره انرژی
- چگونه پنل های خورشیدی ذخیره انرژی هوافضا را باز کنیم
- مجهز به پنل های خورش...

  Record 2:
    ID: 00079aceb2c1667812a082f2a0afa135
    Label: NA sr
    Text preview: ۴ بانوي کوراشکار خراسان رضوي به اردوي تيم ملي راه يافتند
ايرنا/ رئيس انجمن کوراش خراسان رضوي گفت: چه...

  Record 3:
    ID: 000c2fdb3e1d3fbbb3e379e7a2d2fdb2
    Label: 
    Text preview: دانلود آهنگ جدید , دانلود موزیک , دانلود موزیک ویدیو جدید
تمامی مطالب مطابق قوانین جمهوری اسلامی ایر...
Processing: Combined_ID_hybrid.csv


C:\Users\alrazz\AppData\Local\Temp\ipykernel_13236\3797153780.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labels_by_id = df.groupby("id").apply(


  ✓ CSV rows:       8651
  ✓ Unique IDs:     6160
  ✓ Output:         C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\With_na\Combined_ID_hybrid.jsonl

  Sample of first 3 records:

  Record 1:
    ID: 0003f44c9a50428a917087202a2fd3cc
    Label: 
    Text preview: مجموعه های ذخیره انرژی
- چگونه پنل های خورشیدی ذخیره انرژی هوافضا را باز کنیم
- مجهز به پنل های خورش...

  Record 2:
    ID: 00079aceb2c1667812a082f2a0afa135
    Label: NA sr
    Text preview: ۴ بانوي کوراشکار خراسان رضوي به اردوي تيم ملي راه يافتند
ايرنا/ رئيس انجمن کوراش خراسان رضوي گفت: چه...

  Record 3:
    ID: 000c2fdb3e1d3fbbb3e379e7a2d2fdb2
    Label: 
    Text preview: دانلود آهنگ جدید , دانلود موزیک , دانلود موزیک ویدیو جدید
تمامی مطالب مطابق قوانین جمهوری اسلامی ایر...
Processing: Combined_single.csv


C:\Users\alrazz\AppData\Local\Temp\ipykernel_13236\3797153780.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labels_by_id = df.groupby("id").apply(


  ✓ CSV rows:       8255
  ✓ Unique IDs:     6160
  ✓ Output:         C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\With_na\Combined_single.jsonl

  Sample of first 3 records:

  Record 1:
    ID: 0003f44c9a50428a917087202a2fd3cc
    Label: 
    Text preview: مجموعه های ذخیره انرژی
- چگونه پنل های خورشیدی ذخیره انرژی هوافضا را باز کنیم
- مجهز به پنل های خورش...

  Record 2:
    ID: 00079aceb2c1667812a082f2a0afa135
    Label: NA sr
    Text preview: ۴ بانوي کوراشکار خراسان رضوي به اردوي تيم ملي راه يافتند
ايرنا/ رئيس انجمن کوراش خراسان رضوي گفت: چه...

  Record 3:
    ID: 000c2fdb3e1d3fbbb3e379e7a2d2fdb2
    Label: 
    Text preview: دانلود آهنگ جدید , دانلود موزیک , دانلود موزیک ویدیو جدید
تمامی مطالب مطابق قوانین جمهوری اسلامی ایر...
Processing: Combined_SP_hybrid.csv


C:\Users\alrazz\AppData\Local\Temp\ipykernel_13236\3797153780.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labels_by_id = df.groupby("id").apply(


  ✓ CSV rows:       8809
  ✓ Unique IDs:     6160
  ✓ Output:         C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\With_na\Combined_SP_hybrid.jsonl

  Sample of first 3 records:

  Record 1:
    ID: 0003f44c9a50428a917087202a2fd3cc
    Label: 
    Text preview: مجموعه های ذخیره انرژی
- چگونه پنل های خورشیدی ذخیره انرژی هوافضا را باز کنیم
- مجهز به پنل های خورش...

  Record 2:
    ID: 00079aceb2c1667812a082f2a0afa135
    Label: NA sr
    Text preview: ۴ بانوي کوراشکار خراسان رضوي به اردوي تيم ملي راه يافتند
ايرنا/ رئيس انجمن کوراش خراسان رضوي گفت: چه...

  Record 3:
    ID: 000c2fdb3e1d3fbbb3e379e7a2d2fdb2
    Label: 
    Text preview: دانلود آهنگ جدید , دانلود موزیک , دانلود موزیک ویدیو جدید
تمامی مطالب مطابق قوانین جمهوری اسلامی ایر...

Finished processing all CSV files.
JSONL files saved to: C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\With_na


# without NA

In [14]:
import json
import re
from pathlib import Path

import pandas as pd

In [15]:
input_dir = Path(r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\csv_output\without_NA")
output_dir = Path(r"C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\without_NA")

# Create output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)


In [16]:
csv_files = list(input_dir.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files.\n")


for csv_file in csv_files:

    print("=" * 70)
    print(f"Processing: {csv_file.name}")

    try:
        # ----------------------------------------------------
        # Load CSV
        # ----------------------------------------------------
        df = pd.read_csv(
            csv_file,
            keep_default_na=False,
            na_values=[""]
        )

        # ----------------------------------------------------
        # Check required columns
        # ----------------------------------------------------
        required_columns = {
            "id",
            "text",
            "Turku_NLP",
            "Turku_NLP_sub"
        }

        missing_columns = required_columns - set(df.columns)

        if missing_columns:
            print(
                f"  ⚠ Skipping file. Missing columns: "
                f"{sorted(missing_columns)}"
            )
            continue

        # ----------------------------------------------------
        # Consolidate by ID
        # ----------------------------------------------------
        consolidated = (
            df.groupby("id")
            .agg(
                {
                    "text": "first"
                }
            )
            .reset_index()
        )

        # ----------------------------------------------------
        # Add consolidated labels
        # ----------------------------------------------------
        labels_by_id = df.groupby("id").apply(
            consolidate_labels
        )

        consolidated["label"] = (
            consolidated["id"]
            .map(labels_by_id)
            .fillna("")
        )

        # ----------------------------------------------------
        # Create JSONL records
        # ----------------------------------------------------
        records = consolidated.to_dict("records")

        # ----------------------------------------------------
        # Output filename
        #
        # Example:
        # results_Final.CSV
        #     ↓
        # results_Final.jsonl
        # ----------------------------------------------------
        output_file = output_dir / f"{csv_file.stem}.jsonl"

        # ----------------------------------------------------
        # Save JSONL
        # ----------------------------------------------------
        with open(output_file, "w", encoding="utf-8") as f:
            for record in records:
                json.dump(
                    record,
                    f,
                    ensure_ascii=False
                )
                f.write("\n")

        # ----------------------------------------------------
        # Print information
        # ----------------------------------------------------
        print(f"  ✓ CSV rows:       {len(df)}")
        print(f"  ✓ Unique IDs:     {len(consolidated)}")
        print(f"  ✓ Output:         {output_file}")

        # Show first 3 records
        print("\n  Sample of first 3 records:")

        for i, record in enumerate(records[:3]):
            print(f"\n  Record {i + 1}:")
            print(f"    ID: {record['id']}")
            print(f"    Label: {record['label']}")
            print(
                f"    Text preview: "
                f"{str(record['text'])[:100]}..."
            )

    except Exception as e:
        print(f"  ✗ ERROR processing {csv_file.name}: {e}")


print("\n" + "=" * 70)
print("Finished processing all CSV files.")
print(f"JSONL files saved to: {output_dir}")

Found 4 CSV files.

Processing: Combined_hybrid_no_NA.csv


C:\Users\alrazz\AppData\Local\Temp\ipykernel_13236\3797153780.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labels_by_id = df.groupby("id").apply(


  ✓ CSV rows:       7243
  ✓ Unique IDs:     4205
  ✓ Output:         C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\without_NA\Combined_hybrid_no_NA.jsonl

  Sample of first 3 records:

  Record 1:
    ID: 00079aceb2c1667812a082f2a0afa135
    Label: NA sr
    Text preview: ۴ بانوي کوراشکار خراسان رضوي به اردوي تيم ملي راه يافتند
ايرنا/ رئيس انجمن کوراش خراسان رضوي گفت: چه...

  Record 2:
    ID: 0017830cec920da29149e36df490371b
    Label: IP NA ed
    Text preview: حالا 10 سال از وعده اجرای سند ارتقای منزلت سالمندی برای دستیابی به «سالمندی موفق» میگذرد. سوال اساسی...

  Record 3:
    ID: 001f8ba18214e30cb9554bffac1aab8c
    Label: ID IP NA OP ds nb rv
    Text preview: قیمت اعلام شده مربوط به روزهای عادی و تا سقف ظرفیت نرمال میباشد و قیمت در روز های آخر هفته، تعطیلات،...
Processing: Combined_ID_hybrid_no_NA.csv


C:\Users\alrazz\AppData\Local\Temp\ipykernel_13236\3797153780.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labels_by_id = df.groupby("id").apply(


  ✓ CSV rows:       6696
  ✓ Unique IDs:     4205
  ✓ Output:         C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\without_NA\Combined_ID_hybrid_no_NA.jsonl

  Sample of first 3 records:

  Record 1:
    ID: 00079aceb2c1667812a082f2a0afa135
    Label: NA sr
    Text preview: ۴ بانوي کوراشکار خراسان رضوي به اردوي تيم ملي راه يافتند
ايرنا/ رئيس انجمن کوراش خراسان رضوي گفت: چه...

  Record 2:
    ID: 0017830cec920da29149e36df490371b
    Label: IP NA ed
    Text preview: حالا 10 سال از وعده اجرای سند ارتقای منزلت سالمندی برای دستیابی به «سالمندی موفق» میگذرد. سوال اساسی...

  Record 3:
    ID: 001f8ba18214e30cb9554bffac1aab8c
    Label: ID IP NA OP ds nb rv
    Text preview: قیمت اعلام شده مربوط به روزهای عادی و تا سقف ظرفیت نرمال میباشد و قیمت در روز های آخر هفته، تعطیلات،...
Processing: Combined_single_no_NA.csv


C:\Users\alrazz\AppData\Local\Temp\ipykernel_13236\3797153780.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labels_by_id = df.groupby("id").apply(


  ✓ CSV rows:       6300
  ✓ Unique IDs:     4205
  ✓ Output:         C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\without_NA\Combined_single_no_NA.jsonl

  Sample of first 3 records:

  Record 1:
    ID: 00079aceb2c1667812a082f2a0afa135
    Label: NA sr
    Text preview: ۴ بانوي کوراشکار خراسان رضوي به اردوي تيم ملي راه يافتند
ايرنا/ رئيس انجمن کوراش خراسان رضوي گفت: چه...

  Record 2:
    ID: 0017830cec920da29149e36df490371b
    Label: IP NA ed
    Text preview: حالا 10 سال از وعده اجرای سند ارتقای منزلت سالمندی برای دستیابی به «سالمندی موفق» میگذرد. سوال اساسی...

  Record 3:
    ID: 001f8ba18214e30cb9554bffac1aab8c
    Label: ID IP ds
    Text preview: قیمت اعلام شده مربوط به روزهای عادی و تا سقف ظرفیت نرمال میباشد و قیمت در روز های آخر هفته، تعطیلات،...
Processing: Combined_SP_hybrid_no_NA.csv


C:\Users\alrazz\AppData\Local\Temp\ipykernel_13236\3797153780.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  labels_by_id = df.groupby("id").apply(


  ✓ CSV rows:       6854
  ✓ Unique IDs:     4205
  ✓ Output:         C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\without_NA\Combined_SP_hybrid_no_NA.jsonl

  Sample of first 3 records:

  Record 1:
    ID: 00079aceb2c1667812a082f2a0afa135
    Label: NA sr
    Text preview: ۴ بانوي کوراشکار خراسان رضوي به اردوي تيم ملي راه يافتند
ايرنا/ رئيس انجمن کوراش خراسان رضوي گفت: چه...

  Record 2:
    ID: 0017830cec920da29149e36df490371b
    Label: IP NA ed
    Text preview: حالا 10 سال از وعده اجرای سند ارتقای منزلت سالمندی برای دستیابی به «سالمندی موفق» میگذرد. سوال اساسی...

  Record 3:
    ID: 001f8ba18214e30cb9554bffac1aab8c
    Label: ID IP ds
    Text preview: قیمت اعلام شده مربوط به روزهای عادی و تا سقف ظرفیت نرمال میباشد و قیمت در روز های آخر هفته، تعطیلات،...

Finished processing all CSV files.
JSONL files saved to: C:\Users\alrazz\Documents\GitHub\persian_registers\Register\jsonl_output\without_NA
